In [30]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

Read the tickers from a file

In [31]:
#CHANGE TO 'TICKERS.CSV' BY FRIDAY NOV 27
tickers = pd.read_csv('Tickers_Example.csv', header=None)[0].drop_duplicates()

raw_info = {}
for ticker in tickers:
    try:
        raw_info[ticker] = yf.Ticker(ticker).info
    except Exception:
        raw_info[ticker] = {}

In [32]:
type_s= []
currency = []
market = []
sector = []

for ticker in tickers: 
    try: 
        tick = raw_info.get(ticker)
        type_s.append(tick.get('quoteType'))
        currency.append(tick.get('currency'))
        market.append(tick.get('market'))
        sector.append(tick.get('sector'))
    except:
        type_s.append(None)
        currency.append(None)
        market.append(None)
        sector.append(None)

info = {
    'Type' : type_s,
    'Currency' : currency,
    'Market' : market,
    'Sector' : sector
}

all_stocks = pd.DataFrame(info, index=tickers)
all_stocks.index.name = 'Ticker'
all_stocks

,Type,Currency,Market,Sector
Ticker,,,,
AAPL,EQUITY,USD,us_market,Technology
ABBV,EQUITY,USD,us_market,Healthcare
ABT,EQUITY,USD,us_market,Healthcare
ACN,EQUITY,USD,us_market,Technology
AGN,NONE,None,None,None
AIG,EQUITY,USD,us_market,Financial Services
AMZN,EQUITY,USD,us_market,Consumer Cyclical
AXP,EQUITY,USD,us_market,Financial Services
BA,EQUITY,USD,us_market,Industrials


check if it is eligible

In [33]:
mask = all_stocks['Type'] == 'EQUITY'
mask1 = all_stocks['Currency'].isin(['USD', 'CAD'])
mask2 = all_stocks['Market'].isin(['us_market', 'ca_market'])

eligible_stocks = all_stocks[mask & mask1 & mask2]
eligible_stocks

,Type,Currency,Market,Sector
Ticker,,,,
AAPL,EQUITY,USD,us_market,Technology
ABBV,EQUITY,USD,us_market,Healthcare
ABT,EQUITY,USD,us_market,Healthcare
ACN,EQUITY,USD,us_market,Technology
AIG,EQUITY,USD,us_market,Financial Services
AMZN,EQUITY,USD,us_market,Consumer Cyclical
AXP,EQUITY,USD,us_market,Financial Services
BA,EQUITY,USD,us_market,Industrials
BAC,EQUITY,USD,us_market,Financial Services


check for volume

In [34]:
global_start='2025-10-01'
#CHANGE TO '2026-11-27'
global_end='2026-10-28'
tickers_list = eligible_stocks.index.tolist().copy()
hist_data = yf.download(tickers_list, start=global_start, end=global_end, group_by='ticker')

[*********************100%***********************]  35 of 35 completed


In [35]:
start='2025-10-01'
end='2026-10-01'
min_avg_volume=5000

for ticker in eligible_stocks.index.tolist():
    history = hist_data[ticker].loc[start:end]
    history['Month'] = history.index.strftime('%Y-%m')
    
    month_counts = history.groupby('Month').size()
    valid_months = month_counts[month_counts >= 18].index
    history = history[history['Month'].isin(valid_months)]

    eligible_stocks.loc[ticker, 'Avg_Volume'] = history['Volume'].mean()

mask = eligible_stocks['Avg_Volume']>=min_avg_volume
eligible_stocks = eligible_stocks[mask]

eligible_stocks

C:\Users\Polina Lyferenko\AppData\Local\Temp\ipykernel_11588\3791970480.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eligible_stocks.loc[ticker, 'Avg_Volume'] = history['Volume'].mean()


,Type,Currency,Market,Sector,Avg_Volume
Ticker,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07
ACN,EQUITY,USD,us_market,Technology,6.116120e+06
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06
BA,EQUITY,USD,us_market,Industrials,7.140954e+06
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07


take the closing price

In [36]:
#CHANGE DATE LATER TO NOVEMBER 27, 2026
buy_date = '2026-10-27'
end_date = '2026-10-28'
for ticker in eligible_stocks.index.tolist():
    history = hist_data[ticker].loc[buy_date:end_date]
    if not history.empty:
        eligible_stocks.loc[ticker, 'Close'] = history.loc[buy_date, 'Close']
eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume,Close
Ticker,,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06,286.390015
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07,103.099998
ACN,EQUITY,USD,us_market,Technology,6.116120e+06,235.160004
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06,79.580002
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06,312.910004
BA,EQUITY,USD,us_market,Industrials,7.140954e+06,168.179993
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07,50.680000


convert to CAD

In [37]:
usdcad = yf.Ticker('USDCAD=X').history(start=buy_date, end=end_date)
rate = usdcad.loc[buy_date, 'Close']
for ticker in eligible_stocks.index.tolist():
    price = eligible_stocks.loc[ticker, 'Close']
    if eligible_stocks.loc[ticker, 'Currency'] == 'USD':
        eligible_stocks.loc[ticker, 'Close_CAD'] = price * rate
    else :
        eligible_stocks.loc[ticker, 'Close_CAD'] = price

eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD
Ticker,,,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487,486.008102
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06,286.390015,401.306877
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07,103.099998,144.469906
ACN,EQUITY,USD,us_market,Technology,6.116120e+06,235.160004,329.520311
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06,79.580002,111.512275
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006,354.658919
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06,312.910004,438.468277
BA,EQUITY,USD,us_market,Industrials,7.140954e+06,168.179993,235.663900
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07,50.680000,71.015858


fee application function

In [38]:
fee_rate = 0.001 
def calc_fee_cad(shares_num):
    return min(shares_num * fee_rate, 2.15) * rate

calc_fee_cad(10)

np.float64(0.01401260018348694)

PORTFOLIO A

In [42]:
old_date = '2025-10-27'
new_date = '2026-10-28'
for ticker in eligible_stocks.index.tolist():
    history = hist_data[ticker].loc[old_date:new_date]
    eligible_stocks.loc[ticker, 'Old Price'] = history.loc[old_date, 'Close']
    eligible_stocks.loc[ticker, 'New Price'] = history.loc['2026-10-27', 'Close']
    difference = eligible_stocks.loc[ticker, 'New Price']-eligible_stocks.loc[ticker, 'Old Price']
    eligible_stocks.loc[ticker, 'Return %'] = difference/eligible_stocks.loc[ticker, 'Old Price']*100
eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD,Old Price,New Price,Return %
Ticker,,,,,,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487,486.008102,267.617340,346.836487,29.601649
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06,286.390015,401.306877,221.403671,286.390015,29.351972
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07,103.099998,144.469906,124.095634,103.099998,-16.918916
ACN,EQUITY,USD,us_market,Technology,6.116120e+06,235.160004,329.520311,243.751434,235.160004,-3.524669
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06,79.580002,111.512275,77.403282,79.580002,2.812180
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006,354.658919,226.970001,253.100006,11.512537
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06,312.910004,438.468277,357.642914,312.910004,-12.507702
BA,EQUITY,USD,us_market,Industrials,7.140954e+06,168.179993,235.663900,223.000000,168.179993,-24.582963
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07,50.680000,71.015858,51.913418,50.680000,-2.375913


Now sort

In [44]:
eligible_stocks_sorted = eligible_stocks.copy().sort_values(by='Return %', ascending=False)
eligible_stocks_sorted

,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD,Old Price,New Price,Return %
Ticker,,,,,,,,,,
BB.TO,EQUITY,CAD,ca_market,Technology,3.406802e+06,13.390000,13.390000,6.700000,13.390000,99.850757
TXN,EQUITY,USD,us_market,Technology,7.674424e+06,307.409576,430.760749,164.331451,307.409576,87.066793
BIIB,EQUITY,USD,us_market,Healthcare,1.408238e+06,252.729996,354.140438,150.429993,252.729996,68.005058
MRK,EQUITY,USD,us_market,Healthcare,1.143497e+07,141.820007,198.726706,85.489693,141.820007,65.891352
TD.TO,EQUITY,CAD,ca_market,Financial Services,5.614886e+06,176.570007,176.570007,110.914299,176.570007,59.194990
LLY,EQUITY,USD,us_market,Healthcare,3.154767e+06,1280.356201,1794.111954,820.235168,1280.356201,56.096233
BMY,EQUITY,USD,us_market,Healthcare,1.307813e+07,61.549999,86.247553,41.541538,61.549999,48.164950
RY.TO,EQUITY,CAD,ca_market,Financial Services,3.441934e+06,285.459991,285.459991,202.096954,285.459991,41.249032
UNP,EQUITY,USD,us_market,Industrials,3.033148e+06,299.880005,420.209861,213.725555,299.880005,40.310785
